[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inm-unistuttgart/SKHiPPR/blob/tutorial/tutorial/interactive_tutorial.ipynb)

> The badge opens this notebook on Google Colab with nothing to install locally. 

# SKHiPPR in one hour

**S**tability using the **K**oopman-**Hi**ll **P**rojection for **P**eriodic solutions and **R**esonance curves.

This interactive tutorial is written for researchers who already know nonlinear dynamics: equilibria and
their eigenvalues, Floquet theory, harmonic balance, pseudo-arclength continuation. None of
that is explained here. What is explained is how SKHiPPR *says* those things, and how far the
syntax bends before it breaks.

## The whole toolbox in two sentences

1. **An equation is an object whose unknowns are its attributes.** Solving does not return a
   vector; it *mutates* your objects until the residual vanishes.
2. **To gain a degree of freedom, append one equation and one unknown.** Pseudo-arclength
   continuation is that move. So is phase anchoring for autonomous systems. So is whatever you
   write next.

## Roadmap

| § | topic |
|---|---|
| 1 | `Equation` and `EquationSystem`, and why they are different things |
| 2 | Continuation; a `BranchPoint` is an `EquationSystem` with one extra equation |
| 3 | ODEs and DAEs are equations, so they solve for equilibria immediately |
| 4 | `Fourier` and `HBMEquation`: periodic solutions |
| 5 | Stability as a plug-in object you never call |
| 6 | Continuation of periodic solutions — identical code to § 2 |
| 7 | Autonomous systems, DAEs, shooting |
| 8 | Recap and exercises |

The whole notebook runs in about ten seconds.

---
## 0. Setup

Run this cell first. It does one of two things:

* **On Google Colab** — installs `skhippr` straight from GitHub with `pip`. Nothing to clone,
  nothing to configure.
* **On a local checkout** — puts the repository root on `sys.path`, so the notebook works
  whether you launched Jupyter from the repository root or from inside `tutorial/`, and so a
  stale local editable install pointing elsewhere does not shadow this checkout.

SKHiPPR needs **Python 3.13 or newer** (it uses `typing.override` internally). If Colab's
default runtime is older, the cell fails with an assertion telling you what to do — see the
comment in the code below.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

# The branch/tag/commit to install on Colab. Must contain skhippr/odes/daes.py,
# skhippr/odes/autonomous.py and the current visualization API that this notebook uses.
SKHIPPR_GIT_REF = "tutorial"
SKHIPPR_GIT_URL = f"git+https://github.com/inm-unistuttgart/SKHiPPR.git@{SKHIPPR_GIT_REF}"

if IN_COLAB:
    print(f"Google Colab detected (Python {sys.version.split()[0]}).")

    if sys.version_info < (3, 13):
        raise RuntimeError(
            f"This Colab runtime has Python {sys.version.split()[0]}, but SKHiPPR needs "
            "3.13 or newer (it uses typing.override).\n\n"
            "Get a 3.13 runtime with condacolab, then re-run this cell from the top:\n"
            "    !pip install -q condacolab\n"
            "    import condacolab; condacolab.install()   # restarts the runtime once\n\n"
            "After the automatic restart, this same cell will see Python >= 3.13 and "
            "proceed to install skhippr normally. Details: "
            "https://github.com/conda-incubator/condacolab"
        )
    print(f"Installing skhippr by executing in the command line: >> pip install -q {SKHIPPR_GIT_URL}")
    %pip install -q "{SKHIPPR_GIT_URL}"
    print("Installed skhippr from GitHub @", SKHIPPR_GIT_REF)

else:
    import pathlib

    ROOT = pathlib.Path.cwd()
    if not (ROOT / "skhippr").is_dir():          # launched from tutorial/
        ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))

    if "skhippr" in sys.modules:
        print(f"Evicted cached skhippr module from a previous "
                    "import before re-importing from the local checkout.")
        del sys.modules["skhippr"]

    

import numpy as np
import matplotlib.pyplot as plt
from typing import override

%matplotlib inline
plt.rcParams["figure.figsize"] = (6.0, 3.6)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

import pathlib
import skhippr

print("skhippr from:", pathlib.Path(skhippr.__file__).parent)

---
# 1. An equation is an object whose attributes are the unknowns

To define a problem you subclass `AbstractEquation` and implement exactly one method:

* **`residual_function(self)`** — required. Returns a **1-D** `numpy` array computed from
  `self`'s own attributes.
* **`closed_form_derivative(self, variable)`** — optional. Returns
  $\partial r / \partial\,$`variable` as a 2-D array. Raise `NotImplementedError` for any
  variable you do not want to differentiate by hand and SKHiPPR falls back to finite
  differences.

Note that no names of variables are assumed: There is no signature `residual(x)`. The unknowns, as well as any parameters, are attributes,
addressed later by their **name as a string**. That is the whole trick — and the reason a
solved system writes its answer back into your physics object.




We start with the simplest possible example, a point on a circle:

$$ r(\mathbf{y}, \rho) = y_0^2 + y_1^2 - \rho^2 . $$

One scalar residual, two candidate unknowns: $\mathbf{y}$ (1-d numpy array of length 2), $\rho$ (scalar). 

TO DO: Implement the residual function (should return a 1-D numpy array) and its derivatives with respect to `y` (1 x 2 numpy array) and to `radius` (1 x 1 numpy array) as 2-D numpy arrays.

In [ ]:
from skhippr.equations.AbstractEquation import AbstractEquation


class CircleEquation(AbstractEquation):
    """y lies on the circle of radius `radius` around the origin."""

    def __init__(self, y, radius=1.0):
        super().__init__(stability_method=None)
        self.y = y
        self.radius = radius

    @override
    def residual_function(self):
        return ...

    @override
    def closed_form_derivative(self, variable):
        match variable:
            case "y":
                return ...
            case "radius":
                return ...
            case _:
                raise NotImplementedError   # -> finite differences


After having defined the `CircleEquation` class fully, it can be instantiated by giving its attributes arbitrary values. Note that the point `y` defined here does not lie on the circle with radius `radius`. 

TO DO: Instantiate an object of class `CircleEquation`. 

In [ ]:
y = np.array([1.8, 0.0])
radius = 2.0

circle = ...

print("residual       ", circle.residual())
print("d r / d y      ", circle.derivative("y"))
print("d r / d radius ", circle.derivative("radius"))

### The solver

`NewtonSolver` is a *configuration* object (tolerance, iteration cap, verbosity), not a
function. It exposes two entry points:

* `solve_equation(equation, unknown)` — convenience for one equation and one named unknown;
* `solve(equation_system)` — the general case, § 1.2 below.

TO DO: Call `solver.solve_equation` to solve for the appropriate radius that the point `y` lies on and watch what happens to `circle.radius`.

In [ ]:
from skhippr.solvers.newton import NewtonSolver

solver = NewtonSolver(tolerance=1e-10, max_iterations=20, verbose=True)

print(f"before: radius = {circle.radius}, |y| = {np.linalg.norm(circle.y):.4f}\n")

# TO DO: Complete the next line to solve for the radius of the circle given the current y by passing the equation object and the unknown variable name 'radius' as a string to the solver.
solver.solve_equation(equation=..., unknown=...)

print(f"\nafter:  radius = {circle.radius}, |y| = {np.linalg.norm(circle.y):.4f}")
print(f"type:   {type(circle.radius)}   <- scalar unknowns become 1-D arrays")

The solver **edited the object**. There was no return value to unpack — `circle` itself now
holds the solution. Remember this in § 2, where it is what makes a list of branch points a
list of fully-formed solutions.

Also note that the unknown `radius` is now a  1-D numpy array.

## 1.1 One equation, two unknowns: not a problem Newton can take

`y` has two components, the residual has one. Trying to solve for it yields a `ValueError`, and the message
is the entire content of the next section:

In [ ]:
circle.radius = 2.0   # restore

try:
    solver.solve_equation(circle, "y")
except ValueError as err:
    print("ValueError:", err)

To solve for `y` we need a second residual. Let us fix the polar angle:

$$ r_2(\mathbf y, \theta) = y_1\cos\theta - y_0\sin\theta . $$

This second residual is zero if and only if $\mathbf{y}$ encloses an angle of $\theta$ with the $x$ axis.

In [ ]:
class AngleEquation(AbstractEquation):
    """y encloses the angle `theta` with the positive y_0 axis."""

    def __init__(self, y, theta=0.0):
        super().__init__(stability_method=None)
        self.y = y
        self.theta = theta

    @override
    def residual_function(self):
        return np.atleast_1d(
            self.y[1] * np.cos(self.theta) - self.y[0] * np.sin(self.theta)
        )

    @override
    def closed_form_derivative(self, variable):
        match variable:
            case "y":
                return np.atleast_2d(
                    [-np.sin(np.squeeze(self.theta)), np.cos(np.squeeze(self.theta))]
                )
            case "theta":
                return np.atleast_2d(
                    -self.y[1] * np.sin(self.theta) - self.y[0] * np.cos(self.theta)
                )
            case _:
                raise NotImplementedError


angle = AngleEquation(y=circle.y, theta=0.4)

print("residual of circle equation: ", circle.residual())
print("residual of angle equation:  ", angle.residual())

Note that there are now three potential unknown variables: $\mathbf{y}$ as a vector with two components, the scalar radius $\rho$, and the scalar angle $\theta$. 

## 1.2 `EquationSystem` — equations *paired with* unknowns

An `AbstractEquation` has no opinion about what is unknown; `CircleEquation` is equally a
problem in `y` and a problem in `radius`. The object that carries that opinion is the
**`EquationSystem`**: a list of equations together with a list of unknown *names*. It is the
object that can count the number of equations and unknowns, and therefore the only thing a Newton solver can consume.

Below, an equation system is defined by stacking the circle equation and the angle equation, and defining $\mathbf{y}$ (with two entries) as the unknown attribute name to solve for. This yields a well-posed equation system with as many equations as unknowns. Both equations and unknowns get passed into the `EquationSystem` as lists.

The overall residual and Jacobian of the equation system are assembled by stacking the individual residuals and derivatives. 

In [ ]:

from skhippr.equations.EquationSystem import EquationSystem

# TO DO: Complete the next line to create an EquationSystem object that contains the circle and angle equations, and has "y" as the unknown variable.
system = EquationSystem(equations=[...], unknowns=[...])

print("well posed :", system.well_posed, " (2 residuals, 2 unknowns)")
print("residual   :", system.residual_function(update=True))
print("Jacobian   :\n", system.jacobian(update=True))

The unknowns are attributes of the equation system as well as of each underlying equation. Changing the unknown in the equation system propagates to all underlying equations.  

In [ ]:
print("system.y :", system.y)
print("circle.y :", circle.y)
print("angle.y  :", angle.y)


In [ ]:

# TO DO: Change system.y to a new value, e.g., np.array([2.0, 0.1]), and observe the effect on the system and the individual equations.
system.y = ...

print("system.y =", system.y)
print("circle.y =", circle.y, "   angle.y =", angle.y)
print("system.solved  ->", system.solved, " (any assignment invalidates the solution)\n")


Equation systems can be solved by passing them to the `solve()` method of a Newton solver. Note that no parameters (like tolerance) must be passed because they have been passed to the solver during instantiation as attributes. These attributes can be changed at any  time.

TO DO: Check that one iteration is not enough by setting `solver.max_iterations` first to 1 and then to 10 before solving the equation system.

In [ ]:

solver.verbose = True
solver.max_iterations = 1
solver.solve(system)

print("solution       :", system.y)
print("|y|            :", np.linalg.norm(circle.y), "(should be radius =", float(circle.radius), ")")
print("angle          :", np.arctan2(circle.y[1], circle.y[0]), "(should be theta =", angle.theta, ")")
print("solved         :", system.solved)

## 1.3 A note on parameter updates

Unknowns propagate *system → equations*. **Ordinary parameters do not.** A parameter lives on
the equation that uses it, and you change it there:

```python
system.equations[0].radius = 3.0     # right
system.radius = 3.0                  # wrong: sets an unused attribute on the system
```

But an assignment made directly on an equation never passes through the system, so it does not
clear `system.solved` — and `solve()` on an already-solved system returns immediately:

In [ ]:
system.equations[0].radius = 3.0

print("solved flag :", system.solved, "  <- stale!")
solver.solve(system)
print("after solve :", system.y, " |y| =", np.linalg.norm(system.y), " <- still the old answer\n")

# TODO set the attribute `system.solved` to False manually to invalidate the solution before solving again.
...               

solver.solve(system)
print("invalidated :", system.y, " |y| =", np.linalg.norm(system.y), " <- correct")

> **Rule of thumb.** After touching anything *inside* an equation, set `system.solved = False`.
> Inside a continuation loop this is automatic, because every step assigns the unknowns through
> the system.

## 1.4 An equation without writing a class

For quick experiments, `Equation` wraps two plain functions. Its `closed_form_derivative`
callable takes the variable name first; raising `NotImplementedError` selects finite
differences, so this is also the shortest way to see the fallback in action.

In [ ]:
from skhippr.equations.Equation import Equation


def circle_residual(y, radius):
    return np.atleast_1d(y[0] ** 2 + y[1] ** 2 - radius**2)


def no_closed_form(variable, **parameters):
    raise NotImplementedError          # -> finite differences for everything

# TODO complete the instantiation of the Equation class below.
quick_circle = Equation(
    residual_function=...,
    closed_form_derivative=...,
    y=...,            # keyword arguments become attributes
    radius=...,
)

print("residual            :", quick_circle.residual())
print("finite-difference dy:", quick_circle.derivative("y", update=True), "  (exact: [3.6, 0.0])")

solver.solve_equation(quick_circle, "radius")
print("solved radius       :", quick_circle.radius)

### § 1 recap

| you want | you write |
|---|---|
| a residual | subclass `AbstractEquation`, implement `residual_function` |
| an analytic Jacobian | implement `closed_form_derivative`; otherwise raise `NotImplementedError` |
| to declare what is unknown | `EquationSystem(equations=[...], unknowns=["..."])` |
| to solve | `NewtonSolver().solve(system)` — reads and writes your objects |
| to change a parameter | assign on the *equation*, then `system.solved = False` |

---
# 2. Continuation: a branch point is an equation system with one more equation

`pseudo_arclength_continuator` is a **generator**. It takes an `EquationSystem`, solves it, and then yields one `BranchPoint` per converged continuation step, so the loop body — stopping criteria, post-processing, changing parameters mid-branch —
is entirely yours.

It runs in either of two modes:

* **implicit** — the initial system is underdetermined by exactly one equation, such that it admits a 1-dimensional solution curve. The arclength condition provides for each point the missing, last equation. No continuation parameter is named.
* **explicit** — the initial system is already square, and a named `continuation_parameter`
  (any attribute of any equation) is *promoted to an additional unknown*, together with the arclength condition as additional equation.

The circle alone is the implicit case: one residual, two unknowns. Its solution set is exactly
the circle, so continuation should trace it.

In [ ]:
from skhippr.solvers.continuation import pseudo_arclength_continuator
from skhippr.visualization.continuation import plot_continuation

circle = CircleEquation(y=np.array([2.0, 0.0]), radius=2.0)

# TODO: Construct an underdetermined EquationSystem with only the circle equation and "y" as the only unknown variable.
sys_implicit = EquationSystem(...)

print("well posed:", sys_implicit.well_posed, " -> underdetermined, no parameter needed")


We can use the same solver (configuration) as before to solve each continuation step. However, to limit the printouts, let's turn the solver silent:

In [ ]:
solver.verbose = ... # TODO

And now for the actual continuation loop: Stepsize control and the solver are passed to the generator. TO DO: Complete the pseudo-arclength continuation call by passing the previously created initial system and limiting the number of steps to 80.

In every iteration, append the newly returned branch point to the list of branch points. 

In [ ]:

branch = []
for branch_point in pseudo_arclength_continuator(
    initial_system=...,
    solver=solver,
    stepsize=0.2,
    stepsize_range=(0.05, 0.3),
    num_steps=...,
):
    # TODO append the branch_point to the branch point list `branch`.
    ...

print(f"{len(branch)} points, all with |y| = {np.linalg.norm(branch[-1].y):.12f}")

## 2.1 Visualization of continuation results

SKHiPPR comes with quite a few matplotlib visualization options in the `skhippr.visualization` module. To plot continuation results, `plot_continuation` takes the branch and a function `plot_fun` mapping one branch point to one, two or
three scalars:

* **one** scalar → plotted against the continuation parameter (the last unknown);
* **two or three** → used directly as coordinates.

Any optional keyword arguments to `plt.plot()` can be passed to `plot_continuation`. 

Stable segments are drawn in red and unstable in blue, automatically, whenever stability
information is present. Here it is not (the circle has no `stability_method`), so the branch
is black.

In [ ]:
ax = plot_continuation(
    branch,
    plot_fun=lambda bp: bp.y,  # two values -> (x, y) coordinates
    title="Implicit continuation of the circle",
    xlabel="$y_0$",
    ylabel="$y_1$",
)
ax.axis("equal")
plt.show()

In [ ]:
# TODO: Make the same circle, but in blue and with 'x' as markers by copy&pasting the previous cell and adding additional keyword arguments.

## 2.2 Anatomy of a `BranchPoint`

This is the structural heart of SKHiPPR. A `BranchPoint` **is an `EquationSystem`** — the same
class you built by hand in § 1 — carrying:

* every equation of the underlying system, **plus** a `ContinuationAnchor` appended at the end;
* the original unknowns, **plus** the continuation parameter if one was named.

The anchor's residual is identically zero; its *derivative* is the tangent at the previous
point. That is how "the Newton correction must be orthogonal to the tangent" becomes an
ordinary row of the Jacobian rather than special-case solver logic.

In [ ]:
bp = branch[5]

print("bp is an EquationSystem :", isinstance(bp, EquationSystem))
print("equations            :", [e.__class__.__name__ for e in bp.equations])
print("unknowns             :", bp.unknowns)
print("tangent              :", bp.tangent)
print()
print("bp.equations[0]      :", bp.equations[0])
print("  -> your CircleEquation, at this point of the branch:")
print("     y =", bp.equations[0].y, "  radius =", bp.equations[0].radius)
print()
print("branch[0].equations[0] is circle :", branch[0].equations[0] is circle, "(first point shares)")
print("branch[5].equations[0] is circle :", branch[5].equations[0] is circle, "(later points are copies)")

`bp.equations[0]` is the single most useful expression in the library. Because each branch
point owns a duplicate of your equation — already solved, already carrying its stability
information — a list of branch points is a list of complete solutions. 

## 2.3 The loop body is yours

Because the continuator is a generator, you can reach into `branch_point.equations[k]` and
change the problem *while walking the branch*. Growing the radius each time the branch crosses
$y_1 = 0$ turns the circle into a spiral:

In [ ]:
circle_spiral = CircleEquation(y=np.array([2.0, 0.0]), radius=2.0)
spiral = []
y1_prev = 0.0

for branch_point in pseudo_arclength_continuator(
    initial_system=EquationSystem([circle_spiral], unknowns=["y"]),
    solver=solver,
    stepsize=0.2,
    stepsize_range=(0.05, 0.4),
    num_steps=200,
):
    spiral.append(branch_point)

    # TODO: Edit the physics mid-branch by increasing the radius of the circle by 2 when y1 crosses zero from negative to positive. Use the previous y1 value to detect the crossing.

    if y1_prev < 0 <= branch_point.y[1]:
        branch_point.radius = ...      # <- edit the physics mid-branch
    y1_prev = branch_point.y[1]

ax = plot_continuation(spiral, plot_fun=lambda bp: bp.y, title="radius grown from inside the loop",
                       xlabel="$y_0$", ylabel="$y_1$")
ax.axis("equal")
plt.show()

## 2.4 Explicit continuation parameter

Add the angle equation and the system becomes square. Now a parameter must be named — and
*any* attribute of *any* equation will do. 

TO DO: Complete, then run the following code block with continuation parameters `'theta'` and `'radius'` and see the difference:

In [ ]:
# TODO: Instantiate a new CircleEquation and AngleEquation with the same y, and then create an EquationSystem with both equations and "y" as the unknown variable
...
...
system = EquationSystem(...)

continuation_parameter = 'theta' # TODO: Change to 'radius' and observe the difference

branch = []
for bp in pseudo_arclength_continuator(
    initial_system=system,
    solver=solver,
    continuation_parameter=continuation_parameter,
    stepsize=0.1,
    num_steps=400,
):
    branch.append(bp)
    if getattr(bp, continuation_parameter) > 3.5:
        break

branch

ax = plot_continuation(branch, plot_fun=lambda bp: bp.y, color="k", ylabel="$y_1$", xlabel="$y_0$", title=f"continued in {continuation_parameter}")
ax.axis("equal")
plt.show()

---
# 3. ODEs and DAEs are equations, so equilibria are easy

`AbstractODE` subclasses `AbstractEquation` and adds one abstract method,
`dynamics(self, t, x)`. Then:

```python
def residual_function(self):
    return self.dynamics()          # evaluated at self.t, self.x
```

A root of the residual **is** an equilibrium. So an ODE object is already a solvable equation —
no wrapper, no separate "equilibrium problem" class. The constructor also attaches a
`StabilityEquilibrium` method by default, so `ode.stable` and `ode.eigenvalues` are filled in
the moment the solve converges.

The extension contract in full:

| method | required? | notes |
|---|---|---|
| `dynamics(t, x)` | yes | return $\dot x$; optionally write it **vectorised** in `x[i, ...]` so HBM can sample it |
| `closed_form_derivative(variable, t, x)` | no | `NotImplementedError` → finite differences |
| `super().__init__(autonomous, n_dof)` | yes | `autonomous` decides whether HBM needs a phase anchor |

TO DO: Finish the implementation of this complete ODE — the saddle-node normal form $\dot x = \mu - x^2$ — in twelve lines:

In [ ]:
from skhippr.odes.AbstractODE import AbstractODE


class SaddleNode(AbstractODE):
    """dx/dt = mu - x**2"""

    def __init__(self, x, mu):
        super().__init__(autonomous=True, n_dof=1)
        self.t = 0.0
        self.x = x
        self.mu = mu

    @override
    def dynamics(self, t=None, x=None):
        if x is None:
            x = self.x
        self.check_dimensions(t=t, x=x)
        # TODO Add the dynamics here as a 1-D numpy array (tip: x is a 1-D numpy array)
        return ...

    @override
    def closed_form_derivative(self, variable, t=None, x=None):
        if x is None:
            x = self.x
        match variable:
            case "x":
                return np.reshape(-2 * x, (1, 1, *x.shape[1:]))
            case "mu":
                return np.ones((1, 1, *x.shape[1:]))
            case _:
                raise NotImplementedError


ode = SaddleNode(x=np.array([0.7]), mu=1.0)
print("stability method attached automatically:", ode.stability_method)

solver.solve_equation(ode, "x")             # the same call as for the circle

print("equilibrium :", ode.x)
print("eigenvalues :", ode.eigenvalues)
print("stable      :", ode.stable)

## 3.1 Through the fold

The equilibria $x = \pm\sqrt\mu$ meet at a fold at $\mu = 0$, where $\partial f/\partial x = 0$
and naive parameter continuation in $\mu$ must fail. Pseudo-arclength does not care: the
extended Jacobian stays regular. Note that nothing about the loop differs from § 2 — only the
equation inside it.

TO DO: Compute the whole bifurcation diagram for mu between -1 and 1 by copying and modifying the previous continuation call for the circle equation: Only change the system, the continuation parameter, and the breaking criterion. Set the initial_direction argument to -1 to start the continuation decreasing mu.

In [ ]:
eq_sys = EquationSystem(
    equations=[ode], unknowns=["x"], equation_determining_stability=ode
)

branch = [] 

# TODO: Insert continuation here. Set the initial direction to'-1'. 

print(f"{len(branch)} points, {sum(bool(p.stable) for p in branch)} of them stable")
print(f"mu turned around at {min(float(np.squeeze(p.mu)) for p in branch):.4f}")

plot_continuation(
    branch,
    plot_fun=lambda bp: bp.x[0],        # one value -> plotted over mu
    title=r"saddle-node: $\dot x = \mu - x^2$",
    ylabel="$x$",
)
plt.show()

## 3.2 DAEs

`AbstractDAE` subclasses `AbstractODE` for systems $M(t,x)\,\dot x = f(t,x)$ with singular $M$.
For the *equilibrium* problem nothing changes at all: the constraint is simply one more row of
`dynamics`, and the Newton solver never learns that it is special.

The planar pendulum written as a constrained point mass has state
$x = [x, y, \dot x, \dot y, \lambda]$ with the constraint $x^2 + y^2 - \ell^2 = 0$:

In [ ]:
from skhippr.odes.daes import PendulumDAE

dae = PendulumDAE(m=1.0, d=0.1, g=9.81, l=1.0, F=0.0, omega=1.0, phi=0.0)
dae.t = 0.0
dae.x = np.array([0.3, -0.95, 0.0, 0.0, -4.0])   # lambda != 0: keeps df/dx regular

print("residual at the guess:", dae.residual(update=True))

solver.solve_equation(dae, "x")

print("equilibrium          :", np.round(dae.x, 12))
print("residual at the equilibrium:", dae.residual(update=True))
print("expected lambda      :", -1.0 * 9.81 / (2 * 1.0))

The pendulum hangs at $[0, -\ell, 0, 0, -mg/2\ell]$, exactly.

> **Caveat.** `dae.stable` was also computed, but the default `StabilityEquilibrium` method
> eigen-decomposes $\partial f/\partial x$ and ignores the singular mass matrix, so it is *not*
> meaningful for a DAE. Stability of DAEs needs the matrix pencil — which is precisely what the
> DAE-aware Koopman-Hill method in § 7.2 does for periodic solutions.

---
# 4. Periodic solutions: `Fourier` and `HBMEquation`

Two objects, with a clean division of labour.

**`Fourier`** is a *configuration* object, not a solver. It fixes the harmonic truncation and
owns every transform that follows from it:

| attribute | meaning |
|---|---|
| `N_HBM` | highest harmonic retained |
| `L_DFT` | samples per period for the FFT (must be $\ge 2(N_{HBM}+1)$) |
| `n_dof` | states of the signal |
| `real_formulation` | `True`: $[X_0, c_1 \ldots c_N, s_1 \ldots s_N]$; `False`: $[X_{-N} \ldots X_N]$ |

Useful methods: `time_samples`, `DFT`, `inv_DFT`, `derivative_coeffs`, `differentiate`,
`matrix_DFT`, `resize_coefficients`.

In [ ]:
from skhippr.Fourier import Fourier

fourier = Fourier(N_HBM=5, L_DFT=64, n_dof=2, real_formulation=True)
print(fourier)

omega = 2.0
ts = fourier.time_samples(omega=omega)                    # one period, L_DFT samples
x_samp = np.array([np.cos(omega * ts) + 0.5 * np.sin(3 * omega * ts),
                   -omega * np.sin(omega * ts)])

X = fourier.DFT(x_samp)
print("coefficient vector :", X.shape, "= n_dof * (2*N_HBM + 1) =", 2 * (2 * 5 + 1))
print("round trip error   :", np.max(np.abs(fourier.inv_DFT(X) - x_samp)))

dx_exact = np.array([-omega * np.sin(omega * ts) + 1.5 * omega * np.cos(3 * omega * ts),
                     -(omega**2) * np.cos(omega * ts)])
print("spectral derivative:", np.max(np.abs(fourier.differentiate(x_samp, omega) - dx_exact)))

## 4.1 `HBMEquation` — an ODE seen through a `Fourier`

`HBMEquation` is an `AbstractEquation` whose

* **unknown** is `X`, the Fourier coefficient vector of the periodic solution;
* **residual** is the harmonic-balance residual, evaluated by alternating frequency/time:
  transform `X` to samples, call `ode.dynamics(t, x)` on them, transform back, subtract the
  spectral derivative;
* **Jacobian** $\partial R/\partial X$ *is* the Hill matrix, assembled in closed form from
  `ode.closed_form_derivative("x", t, x)`.

Because the residual samples `dynamics` at all `L_DFT` time points at once, writing
`dynamics` vectorised (`x[0, ...]` rather than `x[0]`) is what makes this fast. SKHiPPR falls
back to a Python loop if vectorisation fails.

We use the Duffing oscillator shipped with the package:

$$ \ddot x + \delta\dot x + \alpha x + \beta x^3 = F\cos(\omega t). $$

In [ ]:
from skhippr.odes.nonautonomous import Duffing
from skhippr.cycles.hbm import HBMEquation
from skhippr.stability.KoopmanHillProjection import KoopmanHillSubharmonic

omega0 = 0.3
duffing = Duffing(t=0.0, x=np.array([1.0, 0.0]), omega=omega0,
                  alpha=1.0, beta=2.0, F=0.5, delta=0.16)

fourier = Fourier(N_HBM=15, L_DFT=256, n_dof=2, real_formulation=True)
stability_method = KoopmanHillSubharmonic(fourier, tol=1e-6, autonomous=False)

# initial guess: one harmonic, transformed to the frequency domain
ts = fourier.time_samples(omega0)
X0 = fourier.DFT(np.array([np.cos(omega0 * ts), -omega0 * np.sin(omega0 * ts)]))

hbm = HBMEquation(
    ode=duffing,
    omega=omega0,
    fourier=fourier,
    initial_guess=X0,
    period_k=1,                        # period is period_k times the forcing period
    stability_method=stability_method,
)

print("residual before :", np.linalg.norm(hbm.residual(update=True)))
solver.solve_equation(hbm, "X")        # ... the same call, for the third time
print("residual after  :", np.linalg.norm(hbm.residual(update=False)))
print("Hill matrix     :", hbm.hill_matrix().shape)
print("stable          :", hbm.stable)

### Visualization of periodic solutions

SKHiPPR also allows to visualize periodic solutions (i.e. HBMEquation objects) nicely:



In [ ]:
from skhippr.visualization.cycles import plot_period, plot_phase

plot_period(hbm, num_periods=3, omega=hbm.omega)
plot_phase(hbm)

plt.show()

### Attribute delegation

An `HBMEquation` forwards unknown attribute lookups to its ODE, exactly as an `EquationSystem`
forwards to its equations. `hbm.alpha` *is* `duffing.alpha`; assigning to it changes the ODE.
This is why a continuation parameter can name a physical parameter buried two levels down.

In [ ]:
print("hbm.alpha       :", hbm.alpha, "  (lives on the Duffing object)")
print("hbm.omega       :", hbm.omega)
print("hbm.T           :", hbm.T, "= 2*pi/omega")
print("hbm.T_solution  :", hbm.T_solution, "= period_k * T")
print("x_time()        :", hbm.x_time().shape, "(n_dof x L_DFT samples over one period)")

---
# 5. Stability is a plug-in object you never call

In contrast to stability of equilibria, which is unequivocally given by the (real part of the) eigenvalues of the Jacobian matrix, there are various stability methods to be employed after HBM. For more context, see the dissertation by Fabia Bayer. Every stability method implements one interface:

```python
class AbstractStabilityMethod:
    def determine_eigenvalues(self, equation) -> np.ndarray: ...
    def determine_stability(self, eigenvalues) -> bool: ...
```

You construct one, hand it to the equation, and forget about it. After a successful solve the
equation carries

* **`equation.stable`** — a `bool`,
* **`equation.eigenvalues`** — Floquet multipliers for cycles, Jacobian eigenvalues for
  equilibria.

`EquationSystem.stable` and `.eigenvalues` simply forward to whichever equation was named
`equation_determining_stability`. Which is why § 3's continuation could colour itself.

## The available methods for periodic solutions

| class | idea | cost |
|---|---|---|
| `KoopmanHillProjection` | $\Phi(t) = C\,e^{Ht}\,W$ — direct projection of the Hill matrix exponential, no sorting | one matrix exponential |
| `KoopmanHillSubharmonic` | same, plus a subharmonic copy; error bound decays twice as fast | two matrix exponentials |
| `ClassicalHill` | eigenvalues of the Hill matrix, then *sorting* to pick the true Floquet exponents | one eigendecomposition |
| `SinglePassRK4` / `SinglePassRK38` | explicit Runge-Kutta over one period, reusing the FFT samples | one pass |
| `KoopmanHillDAE`, `KoopmanHillDAESubharmonic` | Koopman-Hill via a Drazin inverse for singular $M$ | § 7.2 |

The Koopman-Hill projection is the method the toolbox is named after: the monodromy matrix is
read off a *single* matrix exponential of the Hill matrix, with no sorting step and with an
explicit, guaranteed convergence bound (Bayer & Leine 2023; Bayer et al. 2024; Bayer & Leine
2025).

Swapping methods on an already-solved cycle costs one assignment:

In [ ]:
import time
from skhippr.stability.KoopmanHillProjection import KoopmanHillProjection
from skhippr.stability.ClassicalHill import ClassicalHill
from skhippr.stability.SinglePass import SinglePassRK4

methods = [
    KoopmanHillProjection(fourier, tol=1e-6),
    KoopmanHillSubharmonic(fourier, tol=1e-6),
    ClassicalHill(fourier, sorting_method="imaginary", tol=1e-6),
    SinglePassRK4(fourier, tol=1e-6),
]

for method in methods:
    hbm.stability_method = method
    t0 = time.perf_counter()
    stable, multipliers = hbm.determine_stability(update=True)
    dt = time.perf_counter() - t0
    print(f"{str(method.label):36s} stable={stable!s:5s} "
          f"|mu| = {np.sort(np.abs(multipliers))}  ({1e3 * dt:5.1f} ms)")

hbm.stability_method = stability_method        # back to the subharmonic method
hbm.determine_stability(update=True)

Four digits of agreement, milliseconds apiece — and the calling code never mentioned Floquet
theory. (The subharmonic projection and the sorted Hill eigenvalue problem agree to eight
digits here; the direct projection and the Runge-Kutta pass bracket them.)

## 5.1 Looking at the stability data

In [ ]:
from skhippr.visualization.cycles import plot_hill_matrix_blocks


plot_hill_matrix_blocks(hbm, logscale=True, cmap="plasma")
plt.title("Hill matrix, blockwise 2-norm")
plt.show()

Each dot is one $n_{dof}\times n_{dof}$ block of the Hill matrix, coloured by its spectral
norm. The decay away from the diagonal is the decay of the Fourier coefficients of
$\partial f/\partial x(x(t))$ — the very quantity in which the Koopman-Hill error bound is
stated (`hbm.exponential_decay_parameters()` extracts it, and
`hbm.error_bound_fundamental_matrix()` turns it into a bound).

In [ ]:
from skhippr.visualization.cycles import plot_floquet_multipliers, plot_floquet_exponents


plot_floquet_multipliers(hbm)
plot_floquet_exponents(hbm)

plt.show()

---
# 6. Continuation of periodic solutions

Here is the promise from § 2, delivered. Compare the loop below with the one that traced a
circle: the equation inside the system changed from `CircleEquation` to `HBMEquation`, and the
continuation parameter from `theta` to `omega`. **Nothing else.**

In [ ]:
frc_system = EquationSystem(
    equations=[hbm], unknowns=["X"], equation_determining_stability=hbm
)

t0 = time.perf_counter()
frc = []
for branch_point in pseudo_arclength_continuator(
    initial_system=frc_system,
    solver=solver,
    continuation_parameter="omega",
    stepsize=0.05,
    stepsize_range=(0.005, 0.1),
    num_steps=2000,
    verbose=False,
):
    frc.append(branch_point)
    if branch_point.omega > 2.5:
        break

print(f"{len(frc)} branch points in {time.perf_counter() - t0:.1f} s; "
      f"{sum(bool(p.stable) for p in frc)} stable")
print("equations per point:", [e.__class__.__name__ for e in frc[-1].equations])
print("unknowns per point :", frc[-1].unknowns)

A `BranchPoint` here holds the `HBMEquation` and a `ContinuationAnchor`, with unknowns
`['X', 'omega']` — the § 2 structure, unchanged.

## 6.1 One branch, three views

The measure plotted is whatever `plot_fun` returns, and `bp.equations[0]` gives it access to
the full solution — here the time series reconstructed from the Fourier coefficients.

In [ ]:
def amplitude(bp):
    return np.max(np.abs(bp.equations[0].x_time()[0, :]))


fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.4))

plot_continuation(frc, plot_fun=amplitude, ax=axs[0])
axs[0].set(title="frequency response", xlabel=r"$\omega$", ylabel=r"$\max_t |x_0|$")

# two return values -> explicit coordinates, so the axes can be swapped
plot_continuation(frc, plot_fun=lambda bp: (amplitude(bp), bp.omega), ax=axs[1])
axs[1].set(title="the same branch, transposed", xlabel=r"$\max_t |x_0|$", ylabel=r"$\omega$")

plt.tight_layout()
plt.show()

In [ ]:
# three return values -> a 3-D branch plot
ax = plot_continuation(
    frc,
    plot_fun=lambda bp: (
        bp.omega,
        amplitude(bp),
        bp.X[bp.equations[0].fourier.n_dof],      # first cosine coefficient of x_0
    ),
    title="frequency response in 3-D",
    xlabel=r"$\omega$",
    ylabel=r"$\max_t|x_0|$",
    zlabel=r"$c_1$",
)
plt.show()

## 6.2 Stability along the branch

The hardening resonance peak overhangs, and the unstable segment between the two folds is
entered and left through a Floquet multiplier crossing $+1$:

In [ ]:
from skhippr.visualization.continuation import (
    plot_floquet_multiplier_continuation,
    plot_floquet_exponent_continuation,
)

fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.4))
plot_floquet_multiplier_continuation(frc, ax=axs[0])
axs[0].axhline(1.0, color="k", lw=1, ls="--")
axs[0].set(title="Floquet multipliers", xlabel=r"$\omega$", ylabel=r"$|\mu_i|$")
plot_floquet_exponent_continuation(frc, ax=axs[1])
axs[1].axhline(0.0, color="k", lw=1, ls="--")
axs[1].set(title="Floquet exponents", xlabel=r"$\omega$", ylabel=r"$\mathrm{Re}\,\alpha_i$")
plt.tight_layout()
plt.show()

## 6.3 Animating the branch

Every `plot_*` function for cycles has an `animate_*` twin that consumes an iterable of
solutions — which is exactly what a branch is. Keep a reference to the returned animation
object or the garbage collector will stop it.

In [ ]:
from IPython.display import HTML
from skhippr.visualization.cycles import animate_phase

ax, animation = animate_phase(frc[::8], interval=120)
ax.set(title="phase portrait along the frequency response")
plt.close(ax.figure)
HTML(animation.to_jshtml())

---
# 7. Versatility

Three variations, each one line of difference from what you have already seen.

## 7.1 Autonomous systems: the same move again

For an autonomous system the frequency is unknown, and the solution is only determined up to a
phase shift. SKHiPPR fixes this the way it fixes everything: **append one equation and one
unknown.** `HBMSystem` does it for you — an `HBMPhaseAnchor` joins the equations and `omega`
joins the unknowns.

Watch the system grow from 1×1 to 2×2, and then to 3×3 once continuation wraps it.

In [ ]:
from skhippr.cycles.hbm import HBMSystem


class Vanderpol(AbstractODE):
    """dx0/dt = x1;  dx1/dt = nu * (1 - x0**2) * x1 - x0"""

    def __init__(self, x, nu, t=0.0):
        super().__init__(autonomous=True, n_dof=2)
        self.x = x
        self.nu = nu
        self.t = t

    @override
    def dynamics(self, t=None, x=None):
        if x is None:
            x = self.x
        self.check_dimensions(x=x)
        f = np.zeros_like(x)
        f[0, ...] = x[1, ...]
        f[1, ...] = self.nu * (1 - x[0, ...] ** 2) * x[1, ...] - x[0, ...]
        return f

    @override
    def closed_form_derivative(self, variable, t=None, x=None):
        if x is None:
            x = self.x
        match variable:
            case "x":
                df = np.zeros((2, *x.shape))
                df[0, 1, ...] = 1
                df[1, 0, ...] = -1 - 2 * self.nu * x[0, ...] * x[1, ...]
                df[1, 1, ...] = self.nu * (1 - x[0, ...] ** 2)
                return df
            case "nu":
                df = np.zeros_like(x)
                df[1, ...] = (1 - x[0, ...] ** 2) * x[1, ...]
                return df[:, np.newaxis, ...]
            case _:
                raise NotImplementedError


vdp_fourier = Fourier(N_HBM=25, L_DFT=512, n_dof=2, real_formulation=True)
vdp = Vanderpol(x=np.array([2.0, 0.0]), nu=0.1)

ts = vdp_fourier.time_samples(1.0)
X0 = vdp_fourier.DFT(np.array([2 * np.cos(ts), -2 * np.sin(ts)]))

vdp_system = HBMSystem(
    ode=vdp,
    omega=1.0,
    fourier=vdp_fourier,
    initial_guess=X0,
    stability_method=KoopmanHillSubharmonic(vdp_fourier, tol=1e-4, autonomous=True),
)

print("equations:", [e.__class__.__name__ for e in vdp_system.equations])
print("unknowns :", vdp_system.unknowns, " <- omega is unknown now")

solver.solve(vdp_system)
print(f"solved: omega = {float(np.squeeze(vdp_system.omega)):.6f}, stable = {vdp_system.stable}")

In [ ]:
vdp_branch = []
for bp in pseudo_arclength_continuator(
    initial_system=vdp_system,
    solver=solver,
    continuation_parameter="nu",
    stepsize=0.05,
    stepsize_range=(0.005, 0.1),
    num_steps=1000,
):
    vdp_branch.append(bp)
    if bp.nu > 2.0:
        break

print("after wrapping in a BranchPoint:")
print("  equations:", [e.__class__.__name__ for e in vdp_branch[-1].equations])
print("  unknowns :", vdp_branch[-1].unknowns)

fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.4))
plot_continuation(vdp_branch, plot_fun=lambda bp: np.max(bp.equations[0].x_time()[1, :]), ax=axs[0])
axs[0].set(title="limit cycle amplitude", xlabel=r"$\nu$", ylabel=r"$\max_t \dot x$")
plot_continuation(vdp_branch, plot_fun=lambda bp: (bp.nu, np.squeeze(bp.omega)), ax=axs[1])
axs[1].set(title="and its frequency", xlabel=r"$\nu$", ylabel=r"$\omega$")
plt.tight_layout()
plt.show()

Three anchors, one pattern:

| situation | appended equation | appended unknown |
|---|---|---|
| continuation | `ContinuationAnchor` (orthogonal to the tangent) | the continuation parameter |
| autonomous HBM | `HBMPhaseAnchor` (phase of one harmonic) | `omega` |
| autonomous shooting | `ShootingPhaseAnchor` (orthogonal to the flow) | `T` |

## 7.2 DAEs: two class names change

For a periodic solution of $M\dot x = f(t, x)$ with singular $M$, swap

* `HBMEquation` → **`HBMEquationDAE`** (the residual accounts for $M$), and
* `KoopmanHillSubharmonic` → **`KoopmanHillDAE`** (the fundamental matrix via a Drazin inverse).

Everything else — solver, unknowns, continuator, plots — is unchanged. As a check, we compute
the *same* forced pendulum twice: once as a 2-state ODE in the angle, once as the 5-state
constrained DAE.

In [ ]:
from skhippr.cycles.hbm import HBMEquationDAE
from skhippr.stability.KoopmanHillProjection import KoopmanHillDAE
from skhippr.odes.daes import PendulumODE

m, g, l, d, F, om, phase = 1.0, 9.81, 1.0, 0.1, 0.5, 1.15, 0.0
N = 10

# --- (a) minimal coordinates: 2-state ODE in the angle -----------------------
pend_ode = PendulumODE(m, d, g, l, F, om, phase)
pend_ode.t, pend_ode.x = 0.0, np.array([0.0, 0.0])

f_ode = Fourier(N_HBM=N, L_DFT=256, n_dof=2, real_formulation=True)
hbm_ode = HBMEquation(
    pend_ode, om, fourier=f_ode,
    initial_guess=np.zeros(2 * (2 * N + 1)),
    stability_method=KoopmanHillSubharmonic(f_ode, tol=1e-4, autonomous=False),
)
solver.solve_equation(hbm_ode, "X")

# --- (b) redundant coordinates + constraint: 5-state DAE ---------------------
phi_t, phi_dot_t = hbm_ode.x_time()

pend_dae = PendulumDAE(m, d, g, l, F, om, phase)
pend_dae.t, pend_dae.x = 0.0, np.zeros(5)

f_dae = Fourier(N_HBM=N, L_DFT=256, n_dof=5, real_formulation=True)
x_dae0 = np.vstack([l * np.sin(phi_t),
                    -l * np.cos(phi_t),
                    l * np.cos(phi_t) * phi_dot_t,
                    l * np.sin(phi_t) * phi_dot_t,
                    np.zeros_like(phi_t)])

hbm_dae = HBMEquationDAE(
    pend_dae, om, fourier=f_dae,
    initial_guess=f_dae.DFT(x_dae0),
    stability_method=KoopmanHillDAE(f_dae, tol=1e-4, autonomous=False, tol_drazin=1e-5),
)
solver.solve_equation(hbm_dae, "X")

print("ODE  |mu| :", np.sort(np.abs(hbm_ode.eigenvalues)))
print("DAE  |mu| :", np.sort(np.abs(hbm_dae.eigenvalues)))
print("           ^ three zeros for the algebraically constrained directions,")
print("             then the two Floquet multipliers of the ODE formulation.")

x_dae = hbm_dae.x_time()
phi_from_dae = np.arctan2(x_dae[0, :], -x_dae[1, :])
print("\nmax |phi_ODE - phi_DAE| :", np.max(np.abs(phi_t - phi_from_dae)))

In [ ]:
t_plot = f_dae.time_samples(om)
fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.4))
axs[0].plot(t_plot, phi_t, label="ODE (angle)")
axs[0].plot(t_plot, phi_from_dae, "--", label="DAE (constrained)")
axs[0].set(title="pendulum angle over one period", xlabel="$t$", ylabel=r"$\varphi$")
axs[0].legend()
axs[1].plot(x_dae[0, :], x_dae[1, :])
axs[1].set(title="DAE states: the constraint is satisfied", xlabel="$x$", ylabel="$y$")
plt.tight_layout()
plt.show()

---
# 8. Recap

**The two sentences.** An equation is an object whose unknowns are its attributes; solving
mutates them. To gain a degree of freedom, append one equation and one unknown.

**To change something, swap one thing:**

| to change | swap |
|---|---|
| the physics | your `AbstractODE` / `AbstractEquation` subclass |
| equilibria → periodic solutions | wrap the ODE in `HBMEquation` (or `ShootingBVP`) |
| ODE → DAE | `HBMEquation` → `HBMEquationDAE`, stability → `KoopmanHillDAE` |
| harmonic truncation | `N_HBM`, `L_DFT` in the `Fourier` object |
| real ↔ complex formulation | `real_formulation` in the `Fourier` object |
| the stability method | the `stability_method` argument — nothing else |
| non-autonomous → autonomous | `HBMEquation` → `HBMSystem` (phase anchor + `omega`) |
| what is continued | the `continuation_parameter` string |
| the nonlinear solver | `NewtonSolver` → `ScipyFsolveSolver` / `ScipyRootSolver` |
| subharmonic / period-$k$ orbits | `period_k` in the `HBMEquation` |

**Where the answers live after a solve:** `equation.residual()`, `equation.derivative(name)`,
`equation.stable`, `equation.eigenvalues`, `hbm.x_time()`, `hbm.hill_matrix()`, and for a
branch, `branch_point.equations[0]`.

## Exercises

1. **Your own oscillator.** Write an `AbstractODE` subclass for a system you care about,
   supply `closed_form_derivative` only for `"x"`, and check the finite-difference fallback by
   comparing `derivative("x")` against `finite_difference_derivative("x")`.
2. **Convergence of the Hill truncation.** Solve the Duffing cycle of § 4 for
   `N_HBM` in `[3, 5, 10, 20, 40]` and plot the Floquet multipliers against $N$. Then repeat
   with `KoopmanHillProjection` instead of `KoopmanHillSubharmonic` and compare the convergence
   rates — the subharmonic error bound decays twice as fast.
3. **A second parameter.** Nest two continuations: sweep `F` in an outer Python loop, run the
   `omega` continuation of § 6 inside it, and plot all branches in one figure to obtain the
   surface of the Duffing resonance.
4. **Period doubling.** Set `period_k=2` in an `HBMEquation` to look for orbits of twice the
   forcing period, and continue a branch emerging near a Floquet multiplier at $-1$.
5. **A new stability method.** Subclass `AbstractStabilityHBM` and implement
   `fundamental_matrix(t_over_period, hbm)` — for instance by direct numerical integration of
   the variational equation. Drop it into the § 5 comparison; nothing else should change.

## References

* Bayer & Leine (2023). *Sorting-free Hill-based stability analysis of periodic solutions
  through Koopman analysis.* **Nonlinear Dyn** 111, 8439–8466.
  <https://doi.org/10.1007/s11071-023-08247-7>
* Bayer, Leine, Thomsen & Brøns (2024). *Koopman-Hill stability computation of periodic orbits
  in polynomial dynamical systems using a real-valued quadratic harmonic balance formulation.*
  **Int. J. Non-Linear Mech.** 167, 104894.
  <https://doi.org/10.1016/j.ijnonlinmec.2024.104894>
* Bayer & Leine (2025, preprint). *Explicit error bounds and guaranteed convergence of the
  Koopman-Hill projection stability method for linear time-periodic dynamics.*
  <https://arxiv.org/abs/2503.21318>
* Documentation: <https://inm-unistuttgart.github.io/SKHiPPR/>